In [ ]:
# Resolve paths consistently from either the repository root or notebooks/.
import os
import sys
from pathlib import Path

PROJECT_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / 'src').is_dir() and (p / 'data').is_dir()), None)
if PROJECT_ROOT is None:
    raise RuntimeError('Run this notebook from within the cloned repository.')
os.chdir(PROJECT_ROOT / 'notebooks')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


# Merge demographic and socioeconomic metrics

In [ ]:
#sets the notebook width
from IPython.display import display, HTML
display(HTML("<style>.container { width:85% !important; }</style>"))
%load_ext autoreload
%autoreload 2

%matplotlib inline
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Standard Libraries
import sys
import os
from pathlib import Path
import glob
import math

# Third party libraries
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats

import statsmodels.api as sm
# from pymc3 import Normal, Model, Uniform,Poisson,Deterministic
# from pymc3 import traceplot,forestplot,summary
# from pymc3 import glm, Model, Metropolis, NUTS, sample
# import pymc3 as pm

# figures
import matplotlib.mlab as ml
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib import ticker
import matplotlib.ticker as plticker

# Local
# from modules import ste

In [ ]:
import matplotlib.mlab as ml
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib import ticker
import matplotlib.ticker as plticker

def stdfigsize(scale=1, nx=1, ny=1, ratio=1.3):
    """
    Returns a tuple to be used as figure size.
    -------
    returns (7*ratio*scale*nx, 7.*scale*ny)
    By default: ratio=1.3
    If ratio<0 them ratio = golden ratio
    """
    if ratio < 0:
        ratio = 1.61803398875
    return((7*ratio*scale*nx, 7*scale*ny))

def stdrcparams(usetex=False):
    """
    Set several mpl.rcParams and sns.set_style for my taste.
    ----
    usetex = True
    ----
    """
    sns.set_style("white")
    sns.set_style({"xtick.direction": "in",
                 "ytick.direction": "in"})
    rcparams = {'text.usetex': usetex,
              'font.family': 'sans-serif',
              'font.sans-serif': ['Times'],
#               'text.latex.unicode': True,
#               'text.latex.preamble': [r"\usepackage[T1]{fontenc}",
#                                       r"\usepackage{lmodern}",
#                                       r"\usepackage{amsmath}",
#                                       r"\usepackage{mathptmx}"
#                                       ],
              'axes.labelsize': 30,
              'axes.titlesize': 30,
              'axes.spines.left':True,
              'axes.spines.bottom':True,
              'axes.spines.top': False,
              'axes.spines.right':False,
              'ytick.left': 'on',
              'xtick.bottom': 'on',
              'xtick.labelsize': '25',
              'ytick.labelsize': '25',
              'axes.linewidth': 1.8,
              'xtick.major.width': 1.8,
              'xtick.minor.width': 1.8,
              'xtick.major.size': 14,
              'xtick.minor.size': 7,
              'xtick.major.pad': 10,
              'xtick.minor.pad': 10,
              'ytick.major.width': 1.8,
              'ytick.minor.width': 1.8,
              'ytick.major.size': 14,
              'ytick.minor.size': 7,
              'ytick.major.pad': 10,
              'ytick.minor.pad': 10,
              'axes.labelpad': 15,
              'axes.titlepad': 15,
              "xtick.direction":
                "out",
              "ytick.direction": "out"
              }
    mpl.rcParams.update(rcparams)

mpl.rcParams['lines.linewidth'] = 5
mpl.rcParams['lines.color'] = '#3690c0'

In [ ]:
stdrcparams(usetex=False)
figsize=stdfigsize(ratio=-1)
xs,ys=figsize
figsize

In [ ]:
min_year = 1960
max_year = 2020

# Load Data

In [ ]:
path = '../data/clean_dataframes/1960-2021/' 

## Poverty 

In [ ]:
poverty_df=pd.read_csv(path+"poverty_population.csv",index_col=0)
poverty_df.head()

In [ ]:
poverty_df[(poverty_df.COUNTY=='Dade County')&(poverty_df.STATE=='Florida')]

In [ ]:
poverty_df[poverty_df.COUNTY.str.contains('Miami')&(poverty_df.STATE=='Florida')]

In [ ]:
fig, ax = plt.subplots(figsize=(xs,ys))
df= poverty_df[[str(i) for i in range(min_year,max_year+1)]]

for city in range(0,len(poverty_df)):
    x=[i for i in range(min_year,max_year+1)]
    y=df.iloc[city]
    ax.plot(x,y,color='k',linewidth=2,alpha=0.6)
# ax.set_yscale('log')
ax.set_ylabel(r"Population",multialignment='center')
ax.set_xlabel(r"Year, $t$")

## Total population

In [ ]:
population_df=pd.read_csv(path+"total_population.csv",index_col=0)
# population_df['FIPS'] = population_df['FIPS'].apply(lambda x: str(int(x)).zfill(5))
population_df.head()

In [ ]:
# population_df[population_df.COUNTY.str.contains('Dade')&(population_df.STATE=='Florida')]

In [ ]:
fig, ax = plt.subplots(figsize=(xs,ys))
df= population_df[[str(i) for i in range(min_year,max_year+1)]]

for city in range(0,len(population_df)):
    x=[i for i in range(min_year,max_year+1)]
    y= df.iloc[city]
    ax.plot(x,y,color='k',linewidth=2,alpha=0.6)
ax.set_ylabel(r"Population",multialignment='center')
ax.set_xlabel(r"Year, $t$")

## Population by race

In [ ]:
black_df=pd.read_csv(path+"black_population.csv",index_col=0)
black_df.head()

In [ ]:
white_df=pd.read_csv(path+"white_population.csv",index_col=0)
white_df.head()

In [ ]:
minorities_df=pd.read_csv(path+"minorities_population.csv",index_col=0)
minorities_df.head()

In [ ]:
nonwhite_df=pd.read_csv(path+"nonwhite_population.csv",index_col=0)
nonwhite_df.head()

In [ ]:
fig, ax = plt.subplots(figsize=(xs,ys))
df= white_df[[str(i) for i in range(min_year,max_year+1)]]

for city in range(0,len(white_df)):
    x=[i for i in range(min_year,max_year+1)]
    y=df.iloc[city]
    ax.plot(x,y,color='k',linewidth=2,alpha=0.6)
ax.set_ylabel(r"Whites",multialignment='center')
ax.set_xlabel(r"Year, $t$")

In [ ]:
fig, ax = plt.subplots(figsize=(xs,ys))
df= nonwhite_df[[str(i) for i in range(min_year,max_year+1)]]

for city in range(0,len(nonwhite_df)):
    x=[i for i in range(min_year,max_year+1)]
    y=df.iloc[city]
    ax.plot(x,y,color='k',linewidth=2,alpha=0.6)
ax.set_ylabel(r"Non-white",multialignment='center')
ax.set_xlabel(r"Year, $t$")

In [ ]:
fig, ax = plt.subplots(figsize=(xs,ys))
df= black_df[[str(i) for i in range(min_year,max_year+1)]]

for city in range(0,len(white_df)):
    x=[i for i in range(min_year,max_year+1)]
    y=df.iloc[city]
    ax.plot(x,y,color='k',linewidth=2,alpha=0.6)
ax.set_ylabel(r"Blacks",multialignment='center')
ax.set_xlabel(r"Year, $t$")

In [ ]:
fig, ax = plt.subplots(figsize=(xs,ys))
df= minorities_df[[str(i) for i in range(min_year,max_year+1)]]

for city in range(0,len(white_df)):
    try:
        x=[i for i in range(min_year,max_year+1)]
        y=df.iloc[city]
        ax.plot(x,y,color='k',linewidth=2,alpha=0.6)
    except:
        continue
ax.set_xlabel(r"Year, $t$")

## Education, and unemployment 

In [ ]:
education_df=pd.read_csv(path+"education.csv",index_col=0)
education_df.head()

In [ ]:
unemployment_df=pd.read_csv(path+"unemployment.csv",index_col=0)
unemployment_df.head()

# Organizing data

In [ ]:
def reorganize_data(dataframe,variable):
    columns=['GISJOIN',    'STATE',  'STATEFP',  'STATENH',   'COUNTY', 'COUNTYFP',
       'COUNTYNH']
    data=[]
    for year in range(min_year,max_year+1):
        df=dataframe[columns].copy()
        df[variable]=dataframe[str(year)]
        df['Year']=[year for i in range(0,len(dataframe))]
        data.append(df)
    df=pd.concat(data)
    df=df.reset_index(drop=True)
    return df

In [ ]:
poverty_df

In [ ]:
poverty_df=reorganize_data(poverty_df,'Poverty')
population_df=reorganize_data(population_df,'Population')
black_df=reorganize_data(black_df,'Blacks')
white_df=reorganize_data(white_df,'Whites')
nonwhite_df=reorganize_data(nonwhite_df,'Non-White')
minorities_df=reorganize_data(minorities_df,'Minorities')
education_df=reorganize_data(education_df,'Education')
unemployment_df=reorganize_data(unemployment_df,'Unemployment')

In [ ]:
full_data=pd.merge(population_df,white_df[['GISJOIN','Year','Whites']],on=['GISJOIN','Year'])
full_data=pd.merge(full_data,poverty_df[['GISJOIN','Year','Poverty']],on=['GISJOIN','Year'])
full_data=pd.merge(full_data,black_df[['GISJOIN','Year','Blacks']],on=['GISJOIN','Year'])
full_data=pd.merge(full_data,education_df[['GISJOIN','Year','Education']],on=['GISJOIN','Year'])
full_data=pd.merge(full_data,unemployment_df[['GISJOIN','Year','Unemployment']],on=['GISJOIN','Year'])
full_data=pd.merge(full_data,minorities_df[['GISJOIN','Year','Minorities']],on=['GISJOIN','Year'])
full_data=pd.merge(full_data,nonwhite_df[['GISJOIN','Year','Non-White']],on=['GISJOIN','Year'])

full_data.head()

In [ ]:
# for i,pop in enumerate(pop_age):
#     full_data=pd.merge(full_data,pop_age[i][['GISJOIN','Year',var_names[i]]],on=['GISJOIN','Year'])

In [ ]:
full_data.head()

In [ ]:
state_dict_df=pd.read_csv('../data/state_abbreviations.csv')
state_dict_df

In [ ]:
state_abbv_dict= dict(state_dict_df.values)

In [ ]:
full_data.STATE = full_data.STATE.map(state_abbv_dict)

In [ ]:
full_data.rename(columns={'STATE':'State','COUNTY':'County','COUNTYFP':'CountyFIPS'},inplace=True)
full_data=full_data[['Year','County','State','CountyFIPS',"STATEFP",'Population', 'Whites', 'Poverty','Blacks','Minorities','Non-White',  'Education', 'Unemployment']]
full_data.head()

In [ ]:
np.unique(full_data[(full_data.State=='Florida')].CountyFIPS)

In [ ]:
full_data[(full_data.CountyFIPS==79)&(full_data.State=='WI')].head()

In [ ]:
full_data[(full_data.State=='NE')].head()

In [ ]:
full_data.columns

In [ ]:
full_data[(full_data.CountyFIPS==86)&(full_data.State=='FL')].head()

In [ ]:
full_data[full_data.Year==2020].shape

In [ ]:
full_data.head()

In [ ]:
full_data.to_csv('../data/clean_dataframes/urban_metrics_interpolated_and_cleaned_1960_2020.csv')